# 20 — Bagging 5-Fold XGBoost (one-hot v2)
## PRT Seguros

Mesma técnica do notebook `19`, mas com XGBoost em vez de CatBoost — para depois blendar os dois
bagging (modelos diferentes = erros diferentes = blend mais eficaz que blendar 2 CatBoost).

## 1. Carregar dados e treinar 5 folds

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb

RANDOM_STATE = 42
ID_COL, TARGET_COL = "cod_individuo", "churned"

train = pd.read_csv("dados_processados/train_model_ready_v2.csv")
val = pd.read_csv("dados_processados/val_model_ready_v2.csv")
kaggle = pd.read_csv("dados_processados/kaggle_model_ready_v2.csv")

treino_completo = pd.concat([train, val], ignore_index=True)
feature_cols = [c for c in treino_completo.columns if c not in (ID_COL, TARGET_COL)]

X_full = treino_completo[feature_cols]
y_full = treino_completo[TARGET_COL]
X_kaggle = kaggle[feature_cols]

MELHORES_PARAMS_XGB = {"subsample": 0.7, "min_child_weight": 5, "max_depth": 4,
                        "learning_rate": 0.02, "gamma": 0.5, "colsample_bytree": 0.7}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_proba = np.zeros(len(X_full))
kaggle_proba_por_fold = []

for fold, (idx_tr, idx_val_fold) in enumerate(skf.split(X_full, y_full)):
    X_tr_fold, y_tr_fold = X_full.iloc[idx_tr], y_full.iloc[idx_tr]
    X_val_fold, y_val_fold = X_full.iloc[idx_val_fold], y_full.iloc[idx_val_fold]

    idx_es_tr, idx_es_val = train_test_split(
        np.arange(len(X_tr_fold)), test_size=0.15, stratify=y_tr_fold, random_state=RANDOM_STATE
    )
    neg_pos_ratio = (y_tr_fold.iloc[idx_es_tr] == 0).sum() / (y_tr_fold.iloc[idx_es_tr] == 1).sum()

    modelo_fold = xgb.XGBClassifier(
        n_estimators=3000, **MELHORES_PARAMS_XGB, scale_pos_weight=neg_pos_ratio,
        tree_method="hist", eval_metric="auc", early_stopping_rounds=50,
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    modelo_fold.fit(
        X_tr_fold.iloc[idx_es_tr], y_tr_fold.iloc[idx_es_tr],
        eval_set=[(X_tr_fold.iloc[idx_es_val], y_tr_fold.iloc[idx_es_val])],
        verbose=False,
    )

    proba_fold = modelo_fold.predict_proba(X_val_fold)[:, 1]
    oof_proba[idx_val_fold] = proba_fold
    auc_fold = roc_auc_score(y_val_fold, proba_fold)

    kaggle_proba_por_fold.append(modelo_fold.predict_proba(X_kaggle)[:, 1])
    print(f"Fold {fold+1}/5 — melhor iteração: {modelo_fold.best_iteration:>4}  AUC-ROC = {auc_fold:.4f}")

auc_oof = roc_auc_score(y_full, oof_proba)
print(f"\nAUC-ROC out-of-fold (XGBoost, one-hot): {auc_oof:.4f}")


Fold 1/5 — melhor iteração:  273  AUC-ROC = 0.8318


Fold 2/5 — melhor iteração:  196  AUC-ROC = 0.8256


Fold 3/5 — melhor iteração:  185  AUC-ROC = 0.8275


Fold 4/5 — melhor iteração:  561  AUC-ROC = 0.8235


Fold 5/5 — melhor iteração:  215  AUC-ROC = 0.8172

AUC-ROC out-of-fold (XGBoost, one-hot): 0.8249


## 2. Salvar previsões (OOF + Kaggle) para o blend final

In [2]:
proba_kaggle_bagging_xgb = np.mean(kaggle_proba_por_fold, axis=0)

pd.DataFrame({"cod_individuo": treino_completo[ID_COL], "proba": oof_proba}).to_csv(
    "dados_processados/proba_val/bagging_xgb_oof.csv", index=False
)
submissao = pd.DataFrame({"Id": kaggle[ID_COL], "probabilidade_churn": proba_kaggle_bagging_xgb})
submissao.to_csv("submissions/submission_bagging_xgb_onehot_5fold.csv", index=False)
print(f"AUC-ROC out-of-fold: {auc_oof:.4f}")
print("Salvo: submissions/submission_bagging_xgb_onehot_5fold.csv")


AUC-ROC out-of-fold: 0.8249
Salvo: submissions/submission_bagging_xgb_onehot_5fold.csv
